# ⚙️ [1/2] Set up your environment

1.  Add Google Colab app to your google drive.
https://www.freecodecamp.org/news/google-colaboratory-python-code-in-your-google-drive/
2.  Run the following command to authorize this notebook to access your Google Drive.

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [2/2] Construct datasets for DiverScan

1.   Input followings, then select `Runtime` -> `Run all`.
2.   When the connection shuts down after all executions are complete, save and then close the notebook.
3.   Go to the `02_search_architecture` notebook.

In [ ]:
#@title ⚙️ 1. General settings

#@markdown 1️⃣ General information

DiverScan_folder = "/content/drive/MyDrive/DiverScan-release" #@param {type:"string"}
data_folder = "/content/drive/MyDrive/DiverScan-release/data/fruitfly" #@param {type:"string"}
project_name = "fruitfly" #@param {type:"string"}
userdefined_location_x = 0.0 #@param {type:"number"}
userdefined_location_y = 0.0 #@param {type:"number"}

#@markdown 2️⃣ Animal and bodypart

data_type = "multi-animal (set number_of_animals >= 2)" #@param ["single (set number_of_animals = 1)", "multi-animal (set number_of_animals >= 2)", "multi-bodypart (set number_of_animals = 1 and number_of_bodyparts >= 2)"]
number_of_animals = 2 #@param {type:"integer"}
number_of_bodyparts = 1 #@param {type:"integer"}
location_of_animal = "Use x and y columns in userset data (set bodyparts_to_calculate_centroid = None)" #@param ["Calculate centroid(s) of animal(s) using skeleton data (set the bodyparts_to_calculate_centroid)", "Use x and y columns in userset data (set bodyparts_to_calculate_centroid = None)", "data_type is 'multi-bodypart' (set bodyparts_to_calculate_centroid = None)"]
bodyparts_to_calculate_centroid = None #@param {type:"raw"}
head_bodypart = 0 #@param {type: "integer"}
tail_bodypart = 0 #@param {type: "integer"}

#@markdown ⚠️ Set `head_bodypart` = 0 and `tail_bodypart` = 0, if `number_of_bodyparts` = 1, `data_type` = multi-bodypart, or just information of head direction is unnecessary.

# The code lives in DiverScan/utils and every project goes under DiverScan/projects.
from pathlib import Path
utils_folder = str(Path(DiverScan_folder) / "utils")
projects_folder = Path(DiverScan_folder) / "projects"


In [ ]:
#@title ⚙️ 2. Features for machine learning

time = False #@param {type: "boolean"}
x = False #@param {type: "boolean"}
y = False #@param {type: "boolean"}
speed = True #@param {type: "boolean"}
azimuth = True #@param {type: "boolean"}
turning_angle = True #@param {type: "boolean"}
acceleration = False #@param {type: "boolean"}
distance_from_userdefined_location = True #@param {type: "boolean"}
angle_from_userdefined_location = True #@param {type: "boolean"}
distance_from_initial_location = True #@param {type: "boolean"}
angle_from_initial_location = True #@param {type: "boolean"}
cumulative_travel_distance = True #@param {type: "boolean"}
cumulative_change_in_turning_angle = True #@param {type: "boolean"}
head_direction = False #@param {type: "boolean"}
userdefined_features = "Yes (set exact column names in the names_of_userdefined_features field)" #@param ["No (set names_of_userdefined_features = None)", "Yes (set exact column names in the names_of_userdefined_features field)"]
names_of_userdefined_features = ['max_wing_ang'] #@param {type:"raw"}

#@markdown ⚠️ If `head_bodypart` = 0 and `tail_bodypart` = 0, do not check `head_direction`.

In [ ]:
#@title ⚙️ 3. Multi mode
#@markdown The single mode ignores followings.

#@markdown ⚙️ Edge index
graph_type = "complete graph (set userset_edges = None)" #@param ["complete graph (set userset_edges = None)", "userset_edges (set your userset_edges)"]
userset_edges = None #@param {type:"raw"}

#@markdown ⚙️ Edge weight
distance_between_nodes = True #@param {type: "boolean"}
azimuth_difference_between_nodes = True #@param {type: "boolean"}
head_direction_difference_between_nodes = False #@param {type: "boolean"}
bodypart_to_bodypart_distance_between_nodes = "No (set bodyparts_pair_list = None)" #@param ["No (set bodyparts_pair_list = None)", "Yes (set the bodyparts_pair_list)"]
bodyparts_pair_list = None #@param {type:"raw"}

#@markdown ⚠️ If `head_bodypart` = 0 and `tail_bodypart` = 0, set `head_direction_difference_between_nodes` to "No".

#@markdown ⚠️ `bodypart_to_bodypart_distance_between_nodes` is for multi-animal ver with `number_of_bodyparts` >= 2. "Yes" expects bi-directional graph. If not, set it to "No".


# Codes

In [ ]:
seed = 0 #@param {type:"integer"}

#@markdown `seed` also fixes the train / val / test split, so the same seed
#@markdown always produces the same split.


In [ ]:
import os
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
from pathlib import Path
from collections import Counter
import pickle
import gzip
import random
import numpy as np
import pandas as pd
import copy

In [ ]:
random.seed(seed)
np.random.seed(seed)

In [ ]:
# Make the DiverScan utils importable. This notebook runs the code in
# DiverScan/utils; the projects it creates get their own copy of it.
import sys

if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import preprocessing as preprocessing_module
import run_record as run_record_module


In [ ]:
np.set_printoptions(formatter={"float": "{:0.6f}".format})

In [ ]:
train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2
set_names = ["train", "val", "test"]

In [ ]:
########## Validate the input values ##########

data_type, bodyparts_list_for_pose = preprocessing_module.validate_input_values(
    data_type=data_type,
    number_of_animals=number_of_animals,
    number_of_bodyparts=number_of_bodyparts,
    location_of_animal=location_of_animal,
    bodyparts_to_calculate_centroid=bodyparts_to_calculate_centroid,
    head_bodypart=head_bodypart,
    tail_bodypart=tail_bodypart,
    head_direction=head_direction,
    userdefined_features=userdefined_features,
    names_of_userdefined_features=names_of_userdefined_features,
    graph_type=graph_type,
    userset_edges=userset_edges,
    head_direction_difference_between_nodes=head_direction_difference_between_nodes,
    bodypart_to_bodypart_distance_between_nodes=bodypart_to_bodypart_distance_between_nodes,
    bodyparts_pair_list=bodyparts_pair_list,
)


In [ ]:
########## Validate the input values - number of features ##########

feature_name_list_for_machine_learning, edge_data_name_list_for_machine_learning = preprocessing_module.build_feature_name_lists(
    data_type=data_type,
    time=time,
    x=x,
    y=y,
    speed=speed,
    azimuth=azimuth,
    turning_angle=turning_angle,
    acceleration=acceleration,
    distance_from_userdefined_location=distance_from_userdefined_location,
    angle_from_userdefined_location=angle_from_userdefined_location,
    distance_from_initial_location=distance_from_initial_location,
    angle_from_initial_location=angle_from_initial_location,
    cumulative_travel_distance=cumulative_travel_distance,
    cumulative_change_in_turning_angle=cumulative_change_in_turning_angle,
    head_direction=head_direction,
    names_of_userdefined_features=names_of_userdefined_features,
    distance_between_nodes=distance_between_nodes,
    azimuth_difference_between_nodes=azimuth_difference_between_nodes,
    head_direction_difference_between_nodes=head_direction_difference_between_nodes,
    bodyparts_pair_list=bodyparts_pair_list,
)


In [ ]:
# Change current directory
os.chdir(data_folder)

In [ ]:
# Get the name of classes
current_path = Path(".")
class_name_list = [
    d.name
    for d in current_path.iterdir()
    if d.is_dir() and d.name != ".ipynb_checkpoints"
]

In [ ]:
########## Validate the folder structure ##########

if len(class_name_list) < 2:
    raise Exception(f'Check the folder structure. Prepare two or more condition folders under your data_folder: {data_folder}')

all_skeleton = []
all_userset = []
for folder in class_name_list:

    skeleton_path = Path(folder) / "skeleton_data_folder"
    userset_path = Path(folder) / "userset_data_folder"

    has_skeleton = skeleton_path.exists()
    has_userset = userset_path.exists()

    all_skeleton.append(has_skeleton)
    all_userset.append(has_userset)

are_folders_empty = not any(all_skeleton) and not any(all_userset)  # To be used later
has_both_of_skeleton_and_userset = all(all_skeleton) and all(all_userset)  # To be used later
has_only_skeleton = all(all_skeleton) and not any(all_userset)  # To be used later
has_only_userset = not any(all_skeleton) and all(all_userset)  # To be used later


if are_folders_empty:
    raise Exception(f'Check the folder structure. Your data_folder does not have any "skeleton_data_folder" or "userset_data_folder". Your data folder: {data_folder}')
else:
    if not (has_both_of_skeleton_and_userset or has_only_skeleton or has_only_userset):
        raise Exception('Check the folder structure. All condition folders in your "data_folder" should contain only "skeleton_data_folder"s, only "userset_data_folder"s, or both of them. EXACTLY NAME "skeleton_data_folder" or "userset_data_folder" them. Your data folder: {data_folder}')

In [ ]:
print(are_folders_empty)
print(has_both_of_skeleton_and_userset)
print(has_only_skeleton)
print(has_only_userset)

In [ ]:
########## Validate CSV files ##########

moving_window_size = preprocessing_module.validate_folders(
    class_name_list,
    data_type,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
    number_of_animals,
    number_of_bodyparts,
)


In [ ]:
########## Validate the input values ##########
if (has_only_userset and data_type == "single") or (has_only_userset and data_type == "multi_animals"):
    if not (number_of_bodyparts == 1):
        raise ValueError('Check "number_of_bodyparts". Set number_of_bodyparts = 1 and include time, x, y columns in your userset_data if you have "userset_data_folder"s but no "skeleton_data_folder"s.')
    if bodyparts_to_calculate_centroid is not None:
        raise ValueError('Check "location_of_animal" and "bodyparts_to_calculate_centroid". Set "Use x and y" and None if you have "userset_data_folder"s but no "skeleton_data_folder"s.')

if has_only_skeleton:
    if names_of_userdefined_features is not None:
        raise ValueError('Check "userdefined_features" and "names_of_userdefined_features". Set "No" and None if you have "skeleton_data_folder"s but not "userset_data_folder"s.')

In [ ]:
########## Create the project folder ##########
project_folder = projects_folder / project_name
counter = 0
while project_folder.exists():
    counter += 1
    project_folder = projects_folder / f"{project_name}_{counter}"

save_folder = project_folder / "dataset"
save_folder.mkdir(parents=True, exist_ok=True)
print("project folder:", project_folder)

# Keep a copy of the code inside the project. The following notebooks import
# from this copy, so the project keeps running with the code it was built with.
utils_snapshot_folder = run_record_module.snapshot_utils(utils_folder, project_folder)
utils_snapshot = utils_snapshot_folder.name


In [ ]:
########## Save settings as a txt file ##########

# Change current directory
os.chdir(save_folder)

settings = {
    "data_folder": data_folder,
    "DiverScan_folder": DiverScan_folder,
    "utils_folder": utils_folder,
    "utils_snapshot": utils_snapshot,
    "project_name": project_name,
    "moving_window_size": moving_window_size,
    "userdefined_location_x": userdefined_location_x,
    "userdefined_location_y": userdefined_location_y,
    "data_type": data_type,
    "number_of_animals": number_of_animals,
    "number_of_bodyparts": number_of_bodyparts,
    "location_of_animal": location_of_animal,
    "bodyparts_to_calculate_centroid": bodyparts_to_calculate_centroid,
    "head_bodypart": head_bodypart,
    "tail_bodypart": tail_bodypart,
    "bodyparts_list_for_pose": bodyparts_list_for_pose,
    "x": x,
    "y": y,
    "speed": speed,
    "azimuth": azimuth,
    "turning_angle": turning_angle,
    "acceleration": acceleration,
    "distance_from_userdefined_location": distance_from_userdefined_location,
    "angle_from_userdefined_location": angle_from_userdefined_location,
    "distance_from_initial_location": distance_from_initial_location,
    "angle_from_initial_location": angle_from_initial_location,
    "cumulative_travel_distance": cumulative_travel_distance,
    "cumulative_change_in_turning_angle": cumulative_change_in_turning_angle,
    "head_direction": head_direction,
    "userdefined_features": userdefined_features,
    "names_of_userdefined_features": names_of_userdefined_features,
    "graph_type": graph_type,
    "userset_edges": userset_edges,
    "distance_between_nodes": distance_between_nodes,
    "azimuth_difference_between_nodes": azimuth_difference_between_nodes,
    "head_direction_difference_between_nodes": head_direction_difference_between_nodes,
    "bodypart_to_bodypart_distance_between_nodes": bodypart_to_bodypart_distance_between_nodes,
    "bodyparts_pair_list": bodyparts_pair_list,
}

with open("settings_of_preprocessing.txt", "a") as file:
    for key, parameter in settings.items():
        file.write(f"{key}: {parameter}\n")

In [ ]:
# Record this notebook and the environment used for this run.
run_record_module.save_run_record(
    utils_folder,
    save_folder,
    notebook_path=Path(DiverScan_folder) / "notebooks" / "01_build_dataset.ipynb",
)


In [ ]:
########## Format the input values ##########

if data_type == "multi_bodyparts":
    bodyparts_to_calculate_centroid = None

if bodyparts_to_calculate_centroid is not None:
    bodyparts_to_calculate_centroid = np.array(bodyparts_to_calculate_centroid) - 1  # Minus 1 from the array to start the bodypart number from zero.

##########

if data_type == "multi_animals" or data_type == "multi_bodyparts":
    if userset_edges is None:
        if data_type == "multi_animals":
             edge_index_one_datapoint = [(int(i), int(j)) for i in range(number_of_animals) for j in range(number_of_animals) if i != j]  # undirected complete graph
        if data_type == "multi_bodyparts":
             edge_index_one_datapoint = [(int(i), int(j)) for i in range(number_of_bodyparts) for j in range(number_of_bodyparts) if i != j]  # undirected complete graph
        edge_index_one_datapoint = np.array(edge_index_one_datapoint).transpose()  # (2, number_of_edges)
    else:
        edge_index_one_datapoint = np.array(userset_edges).transpose() - 1  # (2, number_of_edges). Minus 1 from the array to start the edge number from zero.

##########

head_bodypart = head_bodypart - 1  # Minus 1 to start the bodypart number from zero.
tail_bodypart = tail_bodypart - 1  # Minus 1 to start the bodypart number from zero.
# If head_bodypart = -1 and tail_bodypart = -1, head_direction will not be calculated.

##########

if bodyparts_pair_list:
    bodyparts_pair_list = np.array(bodyparts_pair_list) - 1  # (number of pairs, 2). Minus 1 from the array to start the bodypart number from zero.

##########

In [ ]:
########## Start pre-processing ##########

In [ ]:
from typing import ParamSpecArgs
import math
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import gc

In [ ]:
# Change current directory
os.chdir(data_folder)

classes_dictionary = preprocessing_module.read_csv_files_and_make_onehot_labels(
    class_name_list,
    data_type,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
    )

# Change current directory
os.chdir(save_folder)

dictionary = preprocessing_module.create_train_val_test_set(
    classes_dictionary,
    class_name_list,
    set_names,
    train_ratio,
    val_ratio,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
    split_seed=seed,
)

dictionary, column_name_list = preprocessing_module.format_data(
    dictionary,
    set_names,
    data_type,
    number_of_animals,
    number_of_bodyparts,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
)

threshold_likelihood = None
threshold_distance = None
# Get (time, x, y) data after (x, y) correction based on the thresholds (and calculation of the centroid)
dictionary = preprocessing_module.correct_coords_and_get_time_x_y_data(
    dictionary,
    set_names,
    data_type,
    bodyparts_to_calculate_centroid,
    threshold_likelihood,
    threshold_distance,
    column_name_list,
)

# Convert data from a list to a NumPy array with NaN padding
dictionary = preprocessing_module.convert_from_list_to_array_with_nan_padding(
    dictionary,
    set_names,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
)

In [ ]:
# Delete large-sized data
keys_to_remove = [
    "skeleton_data", "userset_data", "onehot_labels", "skeleton_data_formatted",
    "userset_data_formatted", "skeleton_data_corrected", "time_x_y_data",
]
for set_name in set_names:
    for key in keys_to_remove:
        if key in dictionary[set_name]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]
gc.collect()

In [ ]:
# Calculate features
dictionary, feature_name_list = preprocessing_module.calculate_features(
    dictionary,
    set_names,
    column_name_list,
    has_both_of_skeleton_and_userset,
    has_only_skeleton,
    has_only_userset,
    userdefined_location_x,
    userdefined_location_y,
    head_bodypart,
    tail_bodypart,
)

# Calculate feature for from-skeleton-to-pose convolution
dictionary = preprocessing_module.calculate_pose_feature_convolution(
    dictionary,
    set_names,
    bodyparts_list_for_pose,
)

# Prepare edge_index, edge_data, and edge_data_name_list
if data_type == "single":

    for set_name in set_names:
        number_of_files, *_ = dictionary[set_name]["features"].shape

        dictionary[set_name]["edge_index"] = np.full((number_of_files), None, dtype=object)
        dictionary[set_name]["edge_data"] = np.full((number_of_files), None, dtype=object)

        edge_data_name_list = None

if data_type == "multi_animals" or data_type == "multi_bodyparts":

    dictionary, edge_data_name_list = preprocessing_module.calculate_edge_data(
        dictionary,
        set_names,
        edge_index_one_datapoint,
        feature_name_list,
        bodyparts_pair_list,
    )

# Delete large-sized data
keys_to_remove = [
"userset_nan_padded_array", "time_x_y_nan_padded_array"
]
for set_name in set_names:
    for key in keys_to_remove:
        if key in dictionary[set_name]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]
gc.collect()

In [ ]:
# Create dataset for machine learning
dictionary = preprocessing_module.select_features_for_machine_learning(
    dictionary,
    set_names,
    feature_name_list,
    feature_name_list_for_machine_learning,
    edge_data_name_list,
    edge_data_name_list_for_machine_learning,
)

# Standardization
dictionary, mean_and_std_of_train_dictionary = preprocessing_module.standardize_and_replace_nan_to_zero(
    dictionary,
    set_names,
)

# Delete large-sized data
keys_to_remove = ["features_for_machine_learning", "edge_data_for_machine_learning"]
for set_name in set_names:
    for key in keys_to_remove:
        if key in dictionary[set_name]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]
gc.collect()

# Calculate weighted adjacency matrix
dictionary = preprocessing_module.calculate_weighted_adjacency_matrix(
    dictionary,
    set_names,
)

# Delete large-sized data
keys_to_remove = ["edge_data_standardized"]
for set_name in set_names:
    for key in keys_to_remove:
        if key in dictionary[set_name]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]
gc.collect()

In [ ]:
# Transpose
for set_name in set_names:
    # (Files, N, T, C) -> (Files, C, N, T)
    dictionary[set_name]["features_standardized"] = dictionary[set_name]["features_standardized"].transpose(0, 3, 1, 2)
    # (Files, N, T, C) -> (Files, C, N, T)
    dictionary[set_name]["pose_feature_standardized"] = dictionary[set_name]["pose_feature_standardized"].transpose(0, 3, 1, 2)


# Save
data_names = ["features", "labels", "edge_index", "edge_attributes", "pose_feature"]
keys = ["features_standardized", "onehot_labels_array", "edge_index", "edge_attributes", "pose_feature_standardized"]
train_val_test_dictionary_for_machine_learning = {data_name: {"train": None, "val": None, "test": None} for data_name in data_names}

for set_name in set_names:
    for data_name, key in zip(data_names, keys):
        train_val_test_dictionary_for_machine_learning[data_name][set_name] = dictionary[set_name][key]
        if key in ["features_standardized", "edge_attributes", "pose_feature_standardized"]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]

gc.collect()

data_to_save = {
    "train_val_test_dictionary": train_val_test_dictionary_for_machine_learning,
    "class_name_list": class_name_list,
    "feature_name_list_for_machine_learning": feature_name_list_for_machine_learning,
    "edge_data_name_list_for_machine_learning": edge_data_name_list_for_machine_learning,
    "mean_and_std_of_train_dictionary": mean_and_std_of_train_dictionary,
}

with gzip.open("dataset_standardized_for_machine_learning.pkl", "wb") as file:
    pickle.dump(data_to_save, file)

# Delete large-sized data
for set_name in set_names:
    for data_name in data_names:
        train_val_test_dictionary_for_machine_learning[data_name][set_name] = None
        del train_val_test_dictionary_for_machine_learning[data_name][set_name]

train_val_test_dictionary_for_machine_learning = None
del train_val_test_dictionary_for_machine_learning
gc.collect()

In [ ]:
# Create dataset for post-analysis
dictionary, feature_name_list_for_post_analysis = preprocessing_module.calculate_moving_average_and_variance(
    dictionary,
    ["test"],  # set_names,
    moving_window_size,
    feature_name_list,
)

# Delete large-sized data
keys_to_remove = ["features"]
for set_name in set_names:
    for key in keys_to_remove:
        if key in dictionary[set_name]:
            dictionary[set_name][key] = None
            del dictionary[set_name][key]
gc.collect()


In [ ]:
# Transpose
for set_name in set_names:
    keys = dictionary[set_name].keys()
    for key in keys:
        if type(dictionary[set_name][key]) == np.ndarray and len(dictionary[set_name][key].shape) == 4:
            # (Files, N, T, C) -> (Files, C, N, T)
            dictionary[set_name][key] = dictionary[set_name][key].transpose(0, 3, 1, 2)

# Save for post-analysis. Only the entries the post-analysis reads are kept,
# with the skeleton reduced to (x, y) and the coordinates and edge features
# stored as float32.
data_to_save = {
    "dictionary": preprocessing_module.build_post_analysis_dictionary(dictionary["test"]),
    "class_name_list": class_name_list,
    "feature_name_list": feature_name_list_for_post_analysis,
    "edge_data_name_list": edge_data_name_list,
}

with gzip.open("dataset_unstandardized_for_post_analysis.pkl", "wb") as file:
    pickle.dump(data_to_save, file)

In [ ]:
try:
    from google.colab import runtime
    import time
    time.sleep(5)
    runtime.unassign()
    print("The runtime has been unassigned.")
except ImportError:
    # Not running in Google Colab environment
    pass

# 🟢 Guide
- Prepare CSV files of 2 or more experimental conditions which you want to compare. It is better to prepare over 150 CSV files for one single experimental condition.
- It is strongly recommended to prepare a similar number of CSV files for each condition. ex. 160 files for condition 1 and 150 files for condition 2. If not, DiverScan does not work well due to the "class imbalance" problem.
- The single mode of DiverScan can handle CSV files with different time series lengths, but please avoid analyzing files with extremely different lengths (e.g., more than 10 times longer).
- If you cannot prepare enough files, one option is to increase the number by splitting the original data into smaller clips.

## 🟢 CSV files for single mode

###  Skeleton data from DeepLabCut

           scorer     model      model      model      model      model      model      
           bodyparts  bodypart1  bodypart1  bodypart1  bodypart2  bodypart2  bodypart2  
           coords     x          y          likelihood x          y          likelihood
           0          0.0        0.0        0.99       1.0        1.0        0.99       
           1          0.0        0.0        0.99       1.0        1.0        0.99       
           2          0.0        0.0        0.99       1.0        1.0        0.99       
           3          0.0        0.0        0.99       1.0        1.0        0.99       
           4          0.0        0.0        0.99       1.0        1.0        0.99       
           ...        ...        ...        ...        ...        ...        ...        
           99         0.0        0.0        0.99       1.0        1.0        0.99       

- Prepare CSV files in DeepLabCut format.
- One animal data per one CSV file.
- If you do not have data for likelihood, fill the entire likelihood column with "1."
- Here you can run DiverScan with number_of_animals = 1 and number_of_bodyparts=2.

### Data prepared by other softwares

           time   x    y    longitude   latitude    signal_strength  ...
           0.0    0.5  0.5  139.000000  139.000000  1.0              ...
           1.1    0.5  0.5  139.000000  139.000000  1.1              ...
           2.0    0.5  0.5  139.000000  139.000000  3.2              ...
           3.3    0.5  0.5  139.000000  139.000000  2.2              ...
           4.1    0.5  0.5  139.000000  139.000000  1.0              ...
           ...    ...  ...  ...         ...         ...              ...
          99.0    0.5  0.5  139.000000  139.000000  3.2              ...

- THE COLUMNS MUST BE EXACTLY NAMED "time", "x", and "y" ARE ALWAYS REQUIRED in your CSV files.
- One animal data per one CSV file.
- Do not use the exactly same column names as below "Features for machine learing". eg., Do not use "speed" as the column name.
- Here you can run DiverScan with number_of_animals = 1, number_of_bodyparts=1, and names_of_userdefined_features = ["longitude", "latitude"].
- Any other column can be a feature in the same way. For example, accelerometer channels "acc_x", "acc_y", "acc_z" next to "time", "x", "y" are used alongside the movement features when you set names_of_userdefined_features = ["acc_x", "acc_y", "acc_z"].

🟢 If you also have skeleton data, you can skip "time", "x", and "y".

          longitude   latitude    signal_strength  ...
          139.000000  139.000000  1.0              ...
          139.000000  139.000000  1.1              ...
          139.000000  139.000000  3.2              ...
          139.000000  139.000000  2.2              ...
          139.000000  139.000000  1.0              ...
          ...         ...         ...              ...
          139.000000  139.000000  3.2              ...
⚠️ Ensure that two CSV files (this data and the skeleton data) for the same animal have EXACTLY THE SAME TIME LENGTH AND THE SAME FILE NAME.

## 🟢 CSV files for multi-animal mode

###  Skeleton data from DeepLabCut

           scorer     model        model        model        model        model        model        model        model        model        model        model        model
           bodyparts  bodypart1-1  bodypart1-1  bodypart1-1  bodypart1-2  bodypart1-2  bodypart1-2  bodypart2-1  bodypart2-1  bodypart2-1  bodypart2-2  bodypart2-2  bodypart2-2
           coords     x            y            likelihood   x            y            likelihood   x            y            likelihood   x            y            likelihood
           0          0.0          0.0          0.99         1.0          1.0          0.99         5.0          5.0          0.99         6.0          6.0          0.99
           1          0.0          0.0          0.99         1.0          1.0          0.99         5.0          5.0          0.99         6.0          6.0          0.99
           2          0.0          0.0          0.99         1.0          1.0          0.99         5.0          5.0          0.99         6.0          6.0          0.99
           3          0.0          0.0          0.99         1.0          1.0          0.99         5.0          5.0          0.99         6.0          6.0          0.99
           ...        ...          ...          ...          ...          ...          ...          ...          ...          ...          ...          ...          ...
           99         0.0          0.0          0.99         1.0          1.0          0.99         5.0          5.0          0.99         6.0          6.0          0.99

- Prepare CSV files in DeepLabCut format.
- Multi animals per one CSV file.
- If you do not have data for likelihood, fill the entire likelihood column with "1."
- Here you can run DiverScan with number_of_animals = 2 and number_of_bodyparts = 2.

🟢 The single-animal DeepLabCut format is also acceptable for this mode.

           scorer     model      model      model      model      model      model
           bodyparts  bodypart1  bodypart1  bodypart1  bodypart2  bodypart2  bodypart2
           coords     x          y          likelihood x          y          likelihood
           0          0.0        0.0        0.99       1.0        1.0        0.99
           1          0.0        0.0        0.99       1.0        1.0        0.99
           2          0.0        0.0        0.99       1.0        1.0        0.99
           3          0.0        0.0        0.99       1.0        1.0        0.99
           4          0.0        0.0        0.99       1.0        1.0        0.99
           ...        ...        ...        ...        ...        ...        ...        
           99         0.0        0.0        0.99       1.0        1.0        0.99

- Here you can run DiverScan with number_of_animals = 2 and number_of_bodyparts = 1.

### Data prepared by other softwares

           time   animal1     animal1     animal1     animal1     animal1          animal1     animal2     animal2     animal2     animal2     animal2          animal2
           time   x           y           longitude   latitude    signal_strength  ...         x           y           longitude   latitude    signal_strength  ...
           0.0    0.5         0.5         139.000000  139.000000  1.0              ...         5.5         5.5         139.010000  139.010000  2.3              ...
           1.1    0.5         0.5         139.000000  139.000000  1.1              ...         5.5         5.5         139.010000  139.010000  2.2              ...
           2.0    0.5         0.5         139.000000  139.000000  3.2              ...         5.5         5.5         139.010000  139.010000  1.0              ...
           3.3    0.5         0.5         139.000000  139.000000  2.2              ...         5.5         5.5         139.010000  139.010000  1.0              ...
           4.1    0.5         0.5         139.000000  139.000000  1.0              ...         5.5         5.5         139.010000  139.010000  3.2              ...
           ...    ...         ...         ...         ...         ...              ...         ...         ...         ...         ...         ...              ...
          99.0    0.5         0.5         139.000000  139.000000  3.2              ...         5.5         5.5         139.010000  139.010000  3.3              ...

- THE COLUMNS MUST BE EXACTLY NAMED "time", "x", and "y" ARE ALWAYS REQUIRED in your CSV files. Also, THE FIRST COLUMN MUST BE EXACTRY NAMED "time"(1st row) "time" (second row).
- Multi animals per one CSV file.
- Do not use the exactly same column names as below "Features for machine learing". eg., Do not use "speed" as the column name.
- Here you can run DiverScan with number_of_animals = 2, number_of_bodyparts=1, and names_of_userdefined_features = ["longitude", "latitude"].
- Any other column can be a feature in the same way: give every animal the same extra columns (for example "acc_x", "acc_y", "acc_z") and list them in names_of_userdefined_features.

🟢 If you also have skeleton data, you can skip "time", "x", and "y".

           animal1     animal1     animal1          animal1     animal2     animal2     animal2          animal2
           longitude   latitude    signal_strength  ...         longitude   latitude    signal_strength  ...
           139.000000  139.000000  1.0              ...         139.010000  139.010000  2.3              ...
           139.000000  139.000000  1.1              ...         139.010000  139.010000  2.2              ...
           139.000000  139.000000  3.2              ...         139.010000  139.010000  1.0              ...
           139.000000  139.000000  2.2              ...         139.010000  139.010000  1.0              ...
           139.000000  139.000000  1.0              ...         139.010000  139.010000  3.2              ...
           ...         ...         ...              ...         ...         ...         ...              ...
           139.000000  139.000000  3.2              ...         139.010000  139.010000  3.3              ...

⚠️ Ensure that two CSV files (this data and the skeleton data) for the same animal have EXACTLY THE SAME TIME LENGTH AND THE SAME FILE NAME.

## 🟢 CSV files for multi-bodypart mode

Same as the multi-animal mode in number_of_animals = 1.

## 🟢 File arrangement on your Google Drive

        DiverScan-release
        │
        ├── data
        │        │
        │        └── yourdata_folder (🟢 You can name this folder. Set this folder's path to `data_folder`.)
        │                │
        │                ├── condition_1_folder (🟢 You can name this folder.)
        │                │        │
        │                │        ├── skeleton_data_folder
        │                │        │        ├── ID_1.csv
        │                │        │        ├── ID_2.csv
        │                │        │        ├── ID_3.csv
        │                │        │        └── ...
        │                │        │
        │                │        └── userset_data_folder
        │                │                ├── ID_1.csv
        │                │                ├── ID_2.csv
        │                │                ├── ID_3.csv
        │                │                └── ...
        │                │
        │                ├── condition_2_folder
        │                │        ├── skeleton_data_folder
        │                │        │        ├── ID_101.csv
        │                │        │        ├── ID_102.csv
        │                │        │        ├── ID_103.csv
        │                │        │        └── ...
        │                │        └── userset_data_folder
        │                │                ├── ID_101.csv
        │                │                ├── ID_102.csv
        │                │                ├── ID_103.csv
        │                │                └── ...
        │                │
        │                └── condition_3_folder
        │                        ├── skeleton_data_folder
        │                        │        ├── ID_201.csv
        │                        │        ├── ID_202.csv
        │                        │        ├── ID_203.csv
        │                        │        └── ...
        │                        └── userset_data_folder
        │                                ├── ID_201.csv
        │                                ├── ID_202.csv
        │                                ├── ID_203.csv
        │                                └── ...
        │
        └── utils (🟢 Set this folder's path to `utils_folder`.)
                ├── preprocessing.py
                ├── single_model.py
                ├── multi_model.py
                ├── learning_utils.py
                └── postprocessing.py

                                                                                
You can create as many folders for conditions as you like (at least two).
- ⚠️ Name the folder for CSV files in DeepLabCut format EXACTLY `skeleton_data_folder`. If you do not have them, do not create this folder.
- ⚠️ Name the folder for CSV files prepaired by other softwares EXACTLY `userset_data_folder`. If you do not have them, do not create this folder.
- ⚠️ If you have both of `skeleton_data_folder` and `userset_data_folder`, please ensure that the corresponding files in both folders have EXACTLY THE SAME NAME. (eg., `skeleton_data_folder` > `ID_1.csv`, `userset_data_folder` > `ID_1.csv`)

## 🟢 Note for your local machine

You can run DiverScan on your local machine with GPU. Before running, install following packages:

- torch. See https://pytorch.org/get-started/locally/
- folium  
- IPython  
- ipywidgets  
- joblib  
- matplotlib  
- numpy  
- optuna  
- pandas  
- openpyxl  
- plotly  
- scipy  
- scikit-learn  
- seaborn  
- tqdm